

# Learning Objectives:

- Explore motivation behind using local features
- Understand the requirements for good features
- Understand the general approach of Feature detection and matching pipeline
- Explore each stage of the pipeline with commonly used algorithms


<!-- # Prerequisites: -->

<!-- # Detailed syllabus: -->

<!-- We have discussed briefly about feature detection and matching pipeline already and also explored about one Harris corner detector as a feature detector in the previous chapters. In this chapter, we will

- Explore some widely used approaches for image matching. -->




# Types of features

We discussed features as interest points or regions in images and classes of features in the previous chapter. Generally, two types of image features: global and local features can be extracted from an image.



## Global features:

Global features represent the entire image. It can be described as a property of the entire pixels of the image. Generally, the global feature of an image is a  multi-dimensional feature vector which describes the overall information in the image. For example, global characteristics such as color histograms, texture values, shape parameters, etc.


## Local features:

Local features describe the image patches (small group of pixels). They are generally keypoints or interest regions within an image. An image can contain  multiple local features. Multiple one-dimensional feature vectors represent the entire image. E.g., edges, corners, objects within an image.

The image shown in Figure 1 is represented based on its local and global features.

<center>
<figure>
<img src="https://docs.google.com/uc?export=download&id=17rGh3eR2dEREtzqy7DZuL1PCXm4T0TQb" >
<figcaption align="center">Figure 1: Global vs Local features</figcaption>
</figure>
</center>

# Motivation behind using local features

For image detection and matching tasks, local features are primarily used rather than global features.

- The global features can not distinguish foreground from the background of an image
- The global features also fail to describe a mixture of information from different parts of an image.
- The global features are not robust to occlusions, changes in brightness or viewpoint, image distortions, etc.
- The primary goal of feature detection and matching is to find features in an image that can be found precisely (well-localized) and reliably (well-matched) in other images.
- Local features overcome the limitations found in global features and have high robustness to
  - geometric transformations like rotation, translation
  - occlusions
  - articulation
  - intra-category variations
  - changes in viewpoint or brightness
  - image distortions

# General Approaches to feature detection and matching

## 1. Feature detection (extraction) stage:
In this stage, locations that are likely to match well in other images are searched i.e., we find the interest points in images as shown in Figure 2: (a).

<center>
<figure>
<img src="https://raw.githubusercontent.com/opencv/opencv/master/samples/data/box.png" height=350 width=600>
<figcaption align="center">Figure 2: (a) Feature Detection Stage</figcaption>
</figure>
</center>

## 2. Feature description stage:
In this stage, a region is defined around detected keypoint locations (interest points); the region content is extracted and normalized and converted into compact and stable (invariant) feature descriptors, generally a vector for each feature point as shown in Figure 2: (b).

<center>
<figure>
<img src="https://raw.githubusercontent.com/opencv/opencv/master/samples/data/box_in_scene.png" height=350 width=600>
<figcaption align="center">Figure 2: (b) Feature Description Stage</figcaption>
</figure>
</center>

## 3. Feature matching stage:  
In this stage, we efficiently search for likely matching candidates of feature points in other images as shown in Figure 2: (c)

Feature tracking stage is used as an alternative to feature matching stage that searches a small neighborhood around each detected feature and is therefore more
suitable for video processing.

<center>
<figure>
<img src="https://docs.opencv.org/3.4/matcher_result1.jpg" height=350 width=600>
<figcaption align="center">Figure 2: (c) Feature Matching Stage</figcaption>
</figure>
</center>

## Summary
The general approach of feature detection and matching pipeline discussed above is sumarized in Figure 3.
<center>
<img src="https://raw.githubusercontent.com/opencv/opencv/master/samples/data/box.png" alt="Figure 3: General Approach of feature detection and matching" style="height:350px; width:600px;">
<br>
<em>Figure 3: General Approach of feature detection and matching</em>
</center>

1. Find a set of distinctive interest points in the images.

2. Define a region around each keypoint.

3. Extract and normalize the region content.

4. Compute a local descriptor from the normalized region.

5. Match the features in the images using the local descriptors.



# 1. Feature Detectors



## Features or interest points

We need to find image locations where we can reliably find correspondences with other images for feature detection. We are interested in finding areas where the minimum change caused by shifting the window in any direction is significant.





## Properties of good features:

1. **Saliency:** <br>Features should have a uniqueness, and must be distinct from its immediate neighboring features.
    
    The patches in the image are not salient since they are not distinctive and identifiable. Such features can not match another image of the same scene since they are not distinct.
    
    For example, as shown in Figure 4: (a), if you want to match the feature inside the red rectangle in another image of the same scene, you will not be able to identify the same feature in the next image as almost the entire road has similar patches.

<center>
<figure>
<img src="https://drive.google.com/uc?id=1sTC0TE2SLIc50QItvXVcbxIQKn0KlsLy" width=600>
<figcaption align="center">Figure 4: (a) Repetitive texture are non salient features</figcaption>
</figure>
</center>

2. **Repeatability**:<br> Features should be repeatable i.e., we the same features can be extracted from two different images of the same scene despite geometric and photometric transformations so that we can detect the same point independently in both images. If the points are not repeatable, we cannot match points; hence, image matching fails, as shown in Figure 4: (b).

<center>
<figure>
<img src="https://drive.google.com/uc?id=1JrPV-ILYheGvw8IqWIlpZbJDxLPLaBGS" width=600>
<figcaption align="center">Figure 4: (b) No match since points not repeatable</figcaption>
</figure>
</center>

3. **Locality and Robustness:**<br> Features should be local i.e., occupies a relatively small subset of image space. The features must be robust to occlusion, illumination and brightness variations, clutter, etc.

4. **Quantity**:<br> Features should be abundant in an image i.e., there must be sufficient number of features to represent the object.

5. **Compactness and Efficiency**:<br> Features should be small in number compared to the number of pixels in the image. Also, the feature generation should not have large computation requirements, so that it can be used efficiently in real time applications like self driving cars.

6. **Invariance to geometric transformations**:<br> The features should be locally invariant to geometric transformations like:
    - translation
    - rotation
    - scale
    - affine transformations

   We can observe the locally invariant features in Figure 4: (c). The individual letters are invariant to translation, rotation and scale due to which it can be used for feature matching effectively.
<center>
<figure>
<img src="https://docs.opencv.org/3.4/matcher_result1.jpg" width=600>
<figcaption align="center">Figure 4: (c) Locally Invariant Features (src: <a href='https://docs.opencv.org/3.4/matcher_result1.jpg'>OpenCV Docs</a>)</figcaption>
</figure>
</center>
    



## Examples of good and bad features

Let's demonstrate using the famous Mondrian Painting shown in Figure 5: (a).

<center>
<figure>
<img src="https://docs.google.com/uc?export=download&id=1-DCOy9sa3A8TkRwlGaer53L3dEkbDnrF" width=600>
<figcaption align="center">Figure 5: (a) Mondrian Painting</figcaption>
</figure>
</center>

<center>
<figure>
<img src="https://docs.google.com/uc?export=download&id=1r0gfMa2Z3gYWU-bOpHGEkVrhfZTzDKJ-" width=600>
<figcaption align="center">Figure 5: (b) Patches</figcaption>
</figure>
</center>

Now, let's try to locate the patches in Figure 5: (b), representing different types of features in the painting.

1. **Repetitive texture less patches:**<br> They are the most difficult to localize and are non-salient.  As seen in Figure 6, we cannot identify the small red texture in the painting accurately. They match a lot of areas in the painting. Hence, they are not distinguishable and distinct i.e., they are not repeatable.

    So these are not good feature locations.

<center>
<figure>
<img src="https://docs.google.com/uc?export=download&id=1BQvjUjEShfd3j0GLInycKTyrvYi0lLw9" width=600>
<figcaption align="center">Figure 6: Repetitve texture as bad features</figcaption>
</figure>
</center>

2. **Edges**:<br> They are patches with large contrast change and are easier to localize compared to the repretitive texture patches. However, certain edge regions might still be difficult to localize. As an example, the edge shown in Figure 7, can be matched with numerous edges in the painting. So again, these are challenging locations to use as features, and it isn't easy to get the exact location.

<center>
<figure>
<img src="https://docs.google.com/uc?export=download&id=1Vg8f5UBbksUJYmow2CIZgV5RxN2vENl1" width=600>
<figcaption align="center">Figure 7: Edges as bad features</figcaption>
</figure>
</center>

3. **Corners**:<br> They are the most suitable feature for feature detection and matching pipeline since they are highly lozalized, unique and repeatable. A corner is a feature which has large intensity gradients in two or more directions. An example of a corner is shown in Figure 8.

    It can be seen that the location is easily identified in the painting accurately.

<center>
<figure>
<img src="https://docs.google.com/uc?export=download&id=1fKfS_YvURiNRw6hiiR14Hbg4XNvLD3Sj" width=600>
<figcaption align="center">Figure 8: Corners as good features</figcaption>
</figure>
</center>





## Algorithms for feature detection

There are many existing detectors available such as:

- Harris Corner detector
- Harris-Laplace detector
- Features from accelerated segment test (FAST)
- Laplace of Gaussian (LOG) Detector
- Difference of Gaussian (DOG) Detector

We will be studying about the Harris Corner detector along with the code implementation in the next chapter.



# 2. Feature Descriptors

Once the interest points or features are detected, we need to remember that our end goal is for each point, we need to recognize the corresponding matching point correctly. This is where the feature descriptor comes into play. We need to describe features to allow for feature comparison to determine the best match between image patches. A feature descriptor can be simply thought of as such descriptions of the feature. It can be used to differentiate one feature from another.

<center>
<figure>
<img src="https://docs.google.com/uc?export=download&id=1iLqPb1JnH936oA5gFEDdIkh2vkz9NB67">
<figcaption align="center" width=600>Figure 9: Need for feature descriptor</figcaption>
</figure>
</center>

As seen in Figure 9, we need feature descriptors to match feature points in one image to their corresponding feature point in another image of the same scene or object. Without feature descriptor, features can be detected but cannot be differentiated from one another and hence cannot be matched between the images.

Mathematically, a feature descriptor is an N-dimensional vector $\begin{bmatrix}f_{i,1}\\f_{i,2}\\..\\.. \\f_{i,N} \end{bmatrix}$ assigned to each feature point $\begin{bmatrix}x_i\\y_i\end{bmatrix}$, that provides a summary of the image information around the detected feature.

<center>
<figure>
<img src="https://docs.google.com/uc?export=download&id=11N2WQiTrPjGxz5n7JVswguTO7WvsVbgl" >
<figcaption align="center">Figure 10: Example of keypoint descriptor vectors</figcaption>
</figure>
</center>

## Properties of good feature descriptors:

Similar to feature detectors, feature descriptors also should have some desirable properties to allow for robust feature matching.

1. **Repeatability:** <br> As with feature detectors, descriptors should be repeatable with robustness and invariance to translation, rotation, scale, and illumination changes. This means that regardless of shifts in position, scale, and illumination, the same point of interest in two images should have approximately the same descriptor.

    A large amount of work has been done to provide descriptors that are invariant to scale, illumination, and other image formation variables.
    
2. **Distinctiveness:**<br> Every feature should have distinct feature descriptors. Two nearby features should not have similar descriptors, as this will confuse our feature matching process later on.

3. **Compactness and efficiency:** <br> Generating feature descriptors should be computationaly efficient so that it can be used in applications like autonomous driving which require matching to be performed in real-time.




## Algorithms for feature detectors

- Scale Invariant Feature Transform (SIFT) Descriptor
- Speeded Up Robust Feature (SURF) Descriptor - [Paper](https://link.springer.com/chapter/10.1007/11744023_32), [Article](https://medium.com/data-breach/introduction-to-surf-speeded-up-robust-features-c7396d6e7c4e)
- Binary Robust Invariant Scalable Keypoints (BRISK) Descriptor - [Paper](https://ieeexplore.ieee.org/abstract/document/6126542)
- Binary Robust Independent Elementary Features (BRIEF) Descriptor - [Paper](https://link.springer.com/chapter/10.1007/978-3-642-15561-1_56), [Article](https://medium.com/data-breach/introduction-to-brief-binary-robust-independent-elementary-features-436f4a31a0e6)
- Oriented FAST and Rotated BRIEF (ORB) Descriptor - [Paper](https://www.researchgate.net/publication/221111151_ORB_an_efficient_alternative_to_SIFT_or_SURF), [Article](https://medium.com/data-breach/introduction-to-orb-oriented-fast-and-rotated-brief-4220e8ec40cf)

We will study about SIFT Descriptors along with the implementation in upcoming chapter.  You may explore other descriptors on your own by going through the papers or articles listed above.

# 3. Feature Matching

Before moving to feature matching, let's revise the feature detection and matching pipeline we have discussed.
First, we identified distinctive features or interest points in multiple images using an appropriate feature detector. Then, we calculated a feature descriptor for each extracted feature.

Finally, the last step in the pipeline is to identify the feature matches between the images using the descriptors, which is called as feature matching.

Image matching can be defined as the identification of the correspondences between different images of the same scene or object by comparing feature descriptors or vectors of images to identify their similarities, as shown in Figure 11.

<center>
<figure>
<img src="https://docs.google.com/uc?export=download&id=1IeQFwPmkq_0JsOIgScaOrJo6sChioXOD" width=600>
<figcaption align="center">Figure 11: Feature matching</figcaption>
</figure>
</center>

Image matching can be described as the problem: Given a feature in image $I_1$, how to find the best match in $I_2$?



## Algorithms for image matching:

There are various approaches of image matching. Some of the useful approaches are given below:

- Brute-Force Matcher
- FLANN (Fast Library for Approximate Nearest Neighbors) Matcher

### Brute force matcher

Let's explore the brute force matcher in detail.

It is a simple but powerful feature matching algorithm in which the alignment where most pixels agree is searched. Given a feature and it's a descriptor in image $I_1$, the method proceeds in the following way to find the best match for the feature in image $I_2$.

- Define a distance function that compares the descriptors of two features $f_i$ and $f_j$, by computing the distance between them  The more similar the two descriptors are to each other, the smaller the distance between them.<br>

  The most common distance function used is Sum of Squared Differences (SSD):<br>
    $d(f_i, f_j) = \sum_{k=1}^D(f_{i,k} - f_{j,k})^2$
    <br>

    <!-- Some distance functions used are:
    
    - Sum of Squared Differences (SSD):<br>
    $d(f_i, f_j) = \sum_{k=1}^D(f_{i,k} - f_{j,k})^2$
    <br>

    - Sum of absolute differences (SAD):<br>
    $d(f_i, f_j) = \sum_{k=1}^D\ |f_{i,k} - f_{j,k})|$
    <br>

    - Hamming Distance:<br>
    $d(f_i, f_j) = \sum_{k=1}^D\ XOR(f_{i,k} , f_{j,k})$
    <br>

    where, $D$ is the descriptor size -->

- For every feature $f_i$ in image $I_1$,
    - Apply the distance function $d$ to compute the distance with every feature $f_j$ in image $I_2$.

    - Return the closest match feature $f_c$ from image $I_2$, i.e., the feature with the minimum distance. This feature is called the nearest neighbor and is the closest feature to the original one in the descriptor space.

    - Keep this match only if $d(f_i, f_j)$ is below a predefined threshold $\delta$


<center>
<figure>
<img src="https://docs.google.com/uc?export=download&id=1zcusYxHAbI6j9U-0zQTMCPEjA-WVcZlW">
<figcaption align="center">Figure 12: Brute force matcher</figcaption>
</figure>
</center>

The implementation of Brute Force matcher in OpenCV will be discussed in the SIFT chapter since it requires the computation of feature descriptors before applying the matching algorithm.



# References

1. University Lecture Slides
   * [Juan Carlos Niebles and Ranjay Krishna, Stanford Vision and Learning Lab, Lecture: RANSAC and feature detectors](http://vision.stanford.edu/teaching/cs131_fall1920/slides/05_ransac.pdf)
       * Check pages 24 - 44 to understand overview of feature detection and matching problem

<!-- * Course
    * [Visual Perception for Self-Driving Cars by Steve Waslander, University of Toronto](https://www.coursera.org/learn/visual-perception-self-driving-cars)
        * Check Module 2: Visual Features - Detection, Description and Matching - Lesson 1-3 -->

